# AI_SENTIMENT — Product Review Sentiment Analysis

## Use Case Overview

`AI_SENTIMENT` scores text on a scale from -1 (very negative) to +1 (very positive) using an LLM. Unlike simple keyword matching ("bad" = negative), it understands tone, context, and nuance.

**SA use case:** Instantly surface unhappy customers, identify product issues, and quantify brand health from reviews, tickets, or survey responses — all in SQL.

**Dataset:** 30 synthetic product reviews across 10 product categories. Each review has a known star rating, so we can validate AI_SENTIMENT against a ground truth label.

**What we'll demonstrate:**
1. Basic sentiment scoring on review text
2. Comparing AI_SENTIMENT scores against star ratings
3. Identifying the most negatively-reviewed products
4. Filtering for urgent negative reviews that need attention

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Data Exploration

In [ ]:
df = pd.read_sql("SELECT * FROM product_reviews ORDER BY review_id", conn)
print(f"Total reviews: {len(df)}")
print(f"Products: {df['PRODUCT_NAME'].nunique()}")
print(f"\nStar rating distribution:\n{df['STAR_RATING'].value_counts().sort_index()}")
df[["REVIEW_ID","PRODUCT_NAME","STAR_RATING","REVIEW_TEXT"]].head(5)

## Step 3 — Score Sentiment with AI_SENTIMENT

`AI_SENTIMENT(text)` returns a FLOAT between -1 and +1:
- **+1** → strongly positive
- **0** → neutral
- **-1** → strongly negative

The function runs entirely within Snowflake — no data leaves your account.

In [ ]:
scored_df = pd.read_sql("""
    SELECT
        review_id,
        product_name,
        category,
        star_rating,
        review_text,
        AI_SENTIMENT(review_text) AS sentiment_score
    FROM product_reviews
    ORDER BY review_id
""", conn)
scored_df[["REVIEW_ID","PRODUCT_NAME","STAR_RATING","SENTIMENT_SCORE","REVIEW_TEXT"]].head(10)

## Step 4 — Validate Against Star Ratings

Compare AI_SENTIMENT scores against the known star ratings. A strong positive correlation validates the model is working as expected.

In [ ]:
correlation = scored_df[["STAR_RATING","SENTIMENT_SCORE"]].corr().iloc[0,1]
print(f"Pearson correlation (stars vs sentiment): {correlation:.3f}")

avg_by_stars = scored_df.groupby("STAR_RATING")["SENTIMENT_SCORE"].mean().reset_index()
avg_by_stars.columns = ["star_rating", "avg_sentiment"]
print("\nAverage sentiment score by star rating:")
print(avg_by_stars.to_string(index=False))

avg_by_stars.plot(x="star_rating", y="avg_sentiment", kind="bar",
                  title="Avg Sentiment Score by Star Rating",
                  legend=False, ylabel="Avg Sentiment (-1 to +1)", rot=0)

## Step 5 — Identify Most Negative Products

Aggregate sentiment by product to find which products need the most attention.

In [ ]:
product_sentiment = scored_df.groupby("PRODUCT_NAME").agg(
    review_count=("REVIEW_ID","count"),
    avg_sentiment=("SENTIMENT_SCORE","mean"),
    avg_stars=("STAR_RATING","mean")
).sort_values("avg_sentiment").reset_index()

print("Products ranked by average sentiment (lowest first):")
print(product_sentiment.to_string(index=False))

product_sentiment.plot(x="PRODUCT_NAME", y="avg_sentiment", kind="barh",
                       title="Average Sentiment by Product", legend=False,
                       xlabel="Avg Sentiment Score")

## Step 6 — Interpretation & SA Tips

**Key insight:** AI_SENTIMENT strongly correlates with star ratings but captures nuance that ratings miss. A 3-star review can be net-negative ("decent but...") or net-positive ("good value despite minor issues") — AI_SENTIMENT distinguishes these.

**SA tips:**
- Use `AI_SENTIMENT` in a **Dynamic Table** to continuously score incoming reviews without a pipeline
- Combine with `AI_CLASSIFY` to both score AND categorize in one pass
- A score threshold of `< -0.3` is a good starting point for "needs attention" alerts
- You can feed AI_SENTIMENT output directly into a Streamlit dashboard for real-time product health monitoring